In [1]:
from typing import TypedDict, Annotated
from langchain_core.messages import (HumanMessage, AIMessage, ToolMessage )
from langchain_core.tools import tool
from langgraph.graph import ( StateGraph, START, END)
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_groq import ChatGroq

In [2]:
import os
from dotenv import load_dotenv

# Load API key from backend/.env
dotenv_path = os.path.abspath("../../../../.env")
load_dotenv(dotenv_path)

if not os.environ.get("GROQ_API_KEY"):
    print("Warning: GROQ_API_KEY not found in backend/.env")
else:
    print("GROQ_API_KEY loaded successfully.")

GROQ_API_KEY loaded successfully.


In [3]:

class Contradiction(TypedDict):
    """A contradiction detected between sources."""
    product_slug: str
    field: str
    existing_value: str
    new_value: str
    existing_source: str
    new_source: str
    resolution: str  # pending | resolved
    preferred_source: str

In [4]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [5]:
from typing import TypedDict, Literal, Any
from datetime import datetime

class DBSource(TypedDict):
    """Source item fetched from DB containing category and CSV data."""
    category: str
    data: str  # CSV formatted data string
    format: str  # e.g. "csv"

In [6]:

class LogEntry(TypedDict):
    """A single log entry for log.md."""
    timestamp: str
    operation: str  # ingest | update | lint
    section: str  # knowledge | marketing
    files_processed: list[str]
    products_added: int
    products_updated: int
    reviews_processed: int
    pages_created: list[str]
    pages_updated: list[str]
    conflicts: int
    status: str  # SUCCESS | PARTIAL | FAILED
    errors: list[str]

In [7]:
#  Review State
class ExtractedReview(TypedDict):
    """A review record extracted from a CSV."""
    product_slug: str
    product_name: str
    rating: float
    title: str
    body: str
    reviewer: str
    date: str
    source_file: str

class ReviewSynthesis(TypedDict):
    """Synthesized review data for a product."""
    product_slug: str
    product_name: str
    avg_rating: float
    total_reviews: int
    sentiment_summary: str
    top_pros: list[str]
    top_cons: list[str]
    best_reviews: list[dict[str, str]]    


In [8]:
class WikiPage(TypedDict):
    """Represents a wiki page to create or update."""
    slug: str
    title: str
    page_type: str  # product | category | review | insight | promotion | specialty | popular
    section: str  # knowledge | marketing
    file_path: str
    content: str
    sources: list[str]
    links: list[str]


In [9]:
# Extracted product detail 
class ExtractedProduct(TypedDict):
    """A product entity extracted from a source."""
    name: str
    slug: str  # canonical ID, e.g. "iphone-15"
    brand: str
    category: str
    price: str
    currency: str
    description: str
    specifications: dict[str, str]
    source_file: str
    raw_data: dict[str, Any]

In [10]:

class ValidationResult(TypedDict):
    """Result from wiki validation / lint."""
    orphan_pages: list[str]
    duplicate_products: list[str]
    missing_reviews: list[str]
    conflicting_specs: list[str]
    outdated_prices: list[str]
    broken_links: list[str]
    health_score: float  # 0.0 - 1.0

In [11]:
class WikiState(TypedDict):
    """
    Central state that flows through all LangGraph nodes.

    Both 'knowledge' and 'marketing' sections use this same state.
    The `wiki_section` field controls which sub-directory is targeted.
    """

    # Identity
    merchant_id: str
    wiki_section: str  # "knowledge" | "marketing"

    wiki_base_path: str  # root of merchant_knowledge/

    # DB Data Collection 
    collected_data: list[dict[str, Any]]
    classified_sources: dict[str, list[DBSource]]  # category -> list of DBSource (CSV data)

    extracted_products: list[ExtractedProduct]

    extracted_reviews: list[ExtractedReview]
    review_syntheses: list[ReviewSynthesis]

     # Wiki maintenance 
    entities: list[dict[str, Any]]
    existing_pages: dict[str, str]

    pages_to_update: list[WikiPage]
    contradictions: list[Contradiction]
    pages_to_create: list[WikiPage]
    

    # ── Output ──
    generated_pages: list[WikiPage]
    index_updates: list[dict[str, str]]
    log_entry: LogEntry
    validation_result: ValidationResult

    # ── Control ──
    validation_errors: list[str]
    status: str  # running | success | failed
    error: str


In [12]:
def fetch_db_data(merchant_id: str, section: str) -> list[dict[str, Any]]:
    """
    Database call that fetches N (e.g. 25) raw data rows for a merchant.
    Each row contains the payload data along with its category field 
    (e.g., 'catalog', 'review', 'document', 'promotion').
    """
    # Example SQL / ORM Query:
    # SELECT * FROM merchant_records WHERE merchant_id = :merchant_id AND section = :section
    return []

In [13]:
def collect_data(state: WikiState) -> dict:
    """
    Collects datasets from the DB call (category + CSV data), 
    and directly builds `classified_sources` mapped by category.
    """
    merchant_id = state.get("merchant_id")
    section = state.get("wiki_section", "knowledge")
    if not merchant_id:
        return {
            "collected_data": [],
            "classified_sources": {},
            "status": "failed",
            "error": "Merchant ID is missing from state.",
        }
    try:
        # Fetch category (str) and CSV data directly from DB call
        db_records: list[dict[str, Any]] = fetch_db_data(merchant_id=merchant_id, section=section)
    except Exception as e:
        return {
            "collected_data": [],
            "classified_sources": {},
            "status": "failed",
            "error": f"Database call failed: {str(e)}",
        }
    classified: dict[str, list[dict[str, Any]]] = {}
    for record in db_records:
        category: str = record.get("category", "catalog")
        data_csv: str = record.get("data", "")  # CSV format data string
        source_item = {
            "category": category,
            "data": data_csv,
            "format": "csv",
        }
        if category not in classified:
            classified[category] = []
        classified[category].append(source_item)
    print(f" Collected DB data → Categories: {list(classified.keys())}")
    return {
        "collected_data": db_records,
        "classified_sources": classified,
        "status": "running",
    }


In [14]:
import sys, os
path = os.path.abspath("../")
print(path)
sys.path.insert(0, path)

from utils.llm import call_llm_json

c:\Users\ps302\OneDrive\Desktop\Razorpay\backend\src\agents\knowledge_grap_manager_llm


In [15]:
import json
import csv
import io
import sys, os
path = os.path.abspath("../")
sys.path.insert(0, path)

from utils.llm import call_llm_json
from utils.file_io import slugify

def _parse_csv_data(csv_data: str | list) -> list[dict]:
    """Parse CSV text string into a list of row dictionaries."""
    if isinstance(csv_data, list):
        return csv_data
    if not csv_data or not str(csv_data).strip():
        return []
    
    try:
        reader = csv.DictReader(io.StringIO(csv_data))
        return [dict(row) for row in reader]
    except Exception as e:
        print(f"  Error parsing CSV string: {e}")
        return []


def _extract_from_csv_rows(rows: list[dict], source_name: str = "db_catalog") -> list[ExtractedProduct]:
    """Extract products from CSV rows using LLM in batches of 50."""
    if not rows:
        return []

    products = []

    for batch_start in range(0, len(rows), 50):
        batch = rows[batch_start:batch_start + 50]
        batch_json = json.dumps(batch, default=str, indent=2)

        prompt = (
            f"Extract product entities from these catalog rows.\n\n"
            f"Data:\n{batch_json}\n\n"
            f"For each product, extract:\n"
            f"- name: product name\n"
            f"- brand: brand/manufacturer\n"
            f"- category: product category\n"
            f"- price: price as string\n"
            f"- currency: currency code (default INR)\n"
            f"- description: brief description\n"
            f"- specifications: dict of key-value spec pairs\n\n"
            f"Return JSON: {{\"products\": [{{...}}]}}"
        )

        try:
            result = call_llm_json(prompt, system="You are a product data extractor. Extract structured product information.")
            extracted = result.get("products", [])

            for p in extracted:
                slug = slugify(p.get("name", "unknown"))
                products.append(ExtractedProduct(
                    name=p.get("name", ""),
                    slug=slug,
                    brand=p.get("brand", ""),
                    category=p.get("category", ""),
                    price=str(p.get("price", "")),
                    currency=p.get("currency", "INR"),
                    description=p.get("description", ""),
                    specifications=p.get("specifications", {}),
                    source_file=source_name,
                    raw_data=p,
                ))
        except Exception as e:
            print(f"  Error extracting from CSV batch: {e}")

    return products


def extract_entities(state: WikiState) -> dict:
    """
    Extract product entities from DB data in state (catalog, promotion, documents).
    Deduplicates products by canonical slug.
    """
    classified = state.get("classified_sources", {})
    all_products: list[ExtractedProduct] = []
    seen_slugs: set[str] = set()

    # Target categories that contain product information
    target_categories = ["catalog", "promotion", "documents", "document"]

    for category in target_categories:
        sources = classified.get(category, [])
        for index, source_item in enumerate(sources):
            raw_csv = source_item.get("data", "")
            source_name = f"db_{category}_{index + 1}"

            # Parse CSV string from DB into row dicts
            rows = _parse_csv_data(raw_csv)
            if not rows:
                continue

            # Extract products using LLM batching
            products = _extract_from_csv_rows(rows, source_name=source_name)
            
            for p in products:
                if p["slug"] not in seen_slugs:
                    all_products.append(p)
                    seen_slugs.add(p["slug"])

    print(f"  Extracted {len(all_products)} unique product entities")
    return {"extracted_products": all_products}


In [16]:
"""
extract_reviews — LLM NODE

Parses review CSV data stored in state (fetched from DB) and synthesizes per-product sentiment.
"""

import json
import csv
import io
from collections import defaultdict

from utils.llm import call_llm_json
from utils.file_io import slugify



def _parse_csv_data(csv_data: str | list) -> list[dict]:
    """Parse CSV text string into a list of row dictionaries."""
    if isinstance(csv_data, list):
        return csv_data
    if not csv_data or not str(csv_data).strip():
        return []
    
    try:
        reader = csv.DictReader(io.StringIO(csv_data))
        return [dict(row) for row in reader]
    except Exception as e:
        print(f"  Error parsing CSV string: {e}")
        return []


def _parse_review_rows(rows: list[dict], source_name: str = "db_review") -> list[ExtractedReview]:
    """Parse CSV rows into ExtractedReview objects."""
    reviews = []

    # Normalize column names mapping
    col_map = {}
    if rows:
        sample = rows[0]
        for key in sample.keys():
            key_lower = key.lower().strip()
            if "product" in key_lower and "name" in key_lower:
                col_map["product_name"] = key
            elif "product" in key_lower:
                col_map["product_name"] = key
            elif "name" in key_lower and "product" not in col_map:
                col_map["product_name"] = key
            elif "rating" in key_lower or "star" in key_lower:
                col_map["rating"] = key
            elif "title" in key_lower:
                col_map["title"] = key
            elif "review" in key_lower or "comment" in key_lower or "feedback" in key_lower or "body" in key_lower or "text" in key_lower:
                col_map["body"] = key
            elif "reviewer" in key_lower or "author" in key_lower or "user" in key_lower:
                col_map["reviewer"] = key
            elif "date" in key_lower:
                col_map["date"] = key

    for row in rows:
        product_name = str(row.get(col_map.get("product_name", "product_name"), "Unknown"))
        try:
            rating = float(row.get(col_map.get("rating", "rating"), 0))
        except (ValueError, TypeError):
            rating = 0.0

        reviews.append(ExtractedReview(
            product_slug=slugify(product_name),
            product_name=product_name,
            rating=rating,
            title=str(row.get(col_map.get("title", "title"), "")),
            body=str(row.get(col_map.get("body", "review_text"), "")),
            reviewer=str(row.get(col_map.get("reviewer", "reviewer"), "Anonymous")),
            date=str(row.get(col_map.get("date", "date"), "")),
            source_file=source_name,
        ))

    return reviews


def _synthesize_reviews(product_name: str, product_slug: str, reviews: list[ExtractedReview]) -> ReviewSynthesis:
    """Use LLM to synthesize reviews into sentiment summary."""
    ratings = [r["rating"] for r in reviews if r["rating"] > 0]
    avg_rating = round(sum(ratings) / len(ratings), 1) if ratings else 0.0

    review_texts = []
    for r in reviews[:30]:  # Limit to 30 for token limits
        review_texts.append({
            "rating": r["rating"],
            "title": r["title"],
            "body": r["body"][:300],
            "reviewer": r["reviewer"],
        })

    prompt = (
        f"Synthesize these customer reviews for '{product_name}'.\n\n"
        f"Average rating: {avg_rating}/5 from {len(reviews)} reviews.\n\n"
        f"Reviews:\n{json.dumps(review_texts, indent=2)}\n\n"
        f"Return JSON with:\n"
        f"- sentiment_summary: 2-3 sentence overall sentiment\n"
        f"- top_pros: list of top 5 things customers like\n"
        f"- top_cons: list of top 5 common complaints\n"
        f"- best_reviews: list of 3 most helpful reviews with title, reviewer, rating, excerpt\n\n"
        f"Return JSON: {{\"sentiment_summary\": \"...\", \"top_pros\": [...], \"top_cons\": [...], \"best_reviews\": [...]}}"
    )

    try:
        result = call_llm_json(prompt, system="You are a review analyst. Synthesize customer feedback objectively.")

        return ReviewSynthesis(
            product_slug=product_slug,
            product_name=product_name,
            avg_rating=avg_rating,
            total_reviews=len(reviews),
            sentiment_summary=result.get("sentiment_summary", ""),
            top_pros=result.get("top_pros", []),
            top_cons=result.get("top_cons", []),
            best_reviews=result.get("best_reviews", []),
        )
    except Exception as e:
        print(f"  Error synthesizing reviews for {product_name}: {e}")
        return ReviewSynthesis(
            product_slug=product_slug,
            product_name=product_name,
            avg_rating=avg_rating,
            total_reviews=len(reviews),
            sentiment_summary=f"Based on {len(reviews)} reviews with average rating {avg_rating}/5.",
            top_pros=[],
            top_cons=[],
            best_reviews=[],
        )


def extract_reviews(state: WikiState) -> dict:
    """
    Parse review CSV data from DB state and synthesize per-product sentiment.
    """
    classified = state.get("classified_sources", {})
    all_reviews: list[ExtractedReview] = []

    # Get review sources from DB data in state
    review_sources = classified.get("review", []) + classified.get("reviews", [])

    for index, source_item in enumerate(review_sources):
        raw_csv = source_item.get("data", "")
        source_name = f"db_review_{index + 1}"

        rows = _parse_csv_data(raw_csv)
        if not rows:
            continue

        reviews = _parse_review_rows(rows, source_name=source_name)
        all_reviews.extend(reviews)

    if not all_reviews:
        print("  No reviews found in DB state")
        return {"extracted_reviews": [], "review_syntheses": []}

    # Group reviews by product
    by_product: dict[str, list[ExtractedReview]] = defaultdict(list)
    for review in all_reviews:
        by_product[review["product_slug"]].append(review)

    # Synthesize per product
    syntheses: list[ReviewSynthesis] = []
    for slug, reviews in by_product.items():
        product_name = reviews[0]["product_name"]
        synthesis = _synthesize_reviews(product_name, slug, reviews)
        syntheses.append(synthesis)

    print(f"  Processed {len(all_reviews)} reviews for {len(syntheses)} products")
    return {
        "extracted_reviews": all_reviews,
        "review_syntheses": syntheses,
    }


In [17]:
from utils.file_io import list_wiki_pages, slugify
from utils.wiki_search import fuzzy_match_slug

def search_existing_wiki(state: WikiState) -> dict:
    """
    For each extracted product, check if a wiki page already exists in the repository.
    Populates existing_pages dict: slug → file_path.
    """
    wiki_base = state.get("wiki_base_path", "")
    section = state.get("wiki_section", "knowledge")
    extracted_products = state.get("extracted_products", [])

    if not extracted_products:
        print("  No extracted products to match against existing wiki pages.")
        return {"existing_pages": {}}

    # Get all existing pages across page types in current section
    wiki_dir = f"{wiki_base}/wiki"
    try:
        existing = list_wiki_pages(wiki_dir, section)
    except Exception as e:
        print(f"  Error reading wiki pages from '{wiki_dir}': {e}")
        existing = {}

    # Match extracted products against existing pages
    matched_pages: dict[str, str] = {}

    for product in extracted_products:
        slug = product.get("slug")
        if not slug:
            continue

        # Direct match
        if slug in existing:
            matched_pages[slug] = existing[slug]
            continue

        # Fuzzy match (threshold = 80%)
        fuzzy_results = fuzzy_match_slug(slug, list(existing.keys()), threshold=80)
        if fuzzy_results:
            best_match = fuzzy_results[0]
            matched_pages[slug] = existing[best_match[0]]
            print(f"  Fuzzy matched '{slug}' → '{best_match[0]}' (score: {best_match[1]:.0f})")
            continue

    found = len(matched_pages)
    total = len(extracted_products)
    new_count = total - found
    print(f"  Wiki search: {found} existing pages found, {new_count} new products to create")

    return {"existing_pages": matched_pages}


In [18]:

import json

from utils.file_io import read_markdown, get_page_path
from utils.llm import call_llm_json

def knowledge_diff(state: WikiState) -> dict:
    """
    Diff new extracted info vs existing wiki pages.
    Populates pages_to_create, pages_to_update, and contradictions.
    """
    extracted_products = state.get("extracted_products", [])
    existing_pages = state.get("existing_pages", {})
    review_syntheses = state.get("review_syntheses", [])
    wiki_base = state.get("wiki_base_path", "")
    section = state.get("wiki_section", "knowledge")

    pages_to_create: list[WikiPage] = []
    pages_to_update: list[WikiPage] = []
    contradictions: list[Contradiction] = []

    for product in extracted_products:
        slug = product.get("slug")
        if not slug:
            continue

        name = product.get("name", "Unknown Product")
        source_id = product.get("source_file", "db_source")
        page_type = "products"

        # Determine page type for marketing section
        if section == "marketing":
            raw_data = str(product.get("raw_data", {})).lower()
            if any(k in raw_data for k in ["promotion", "discount", "offer", "deal"]):
                page_type = "promotions"
            elif any(k in raw_data for k in ["special", "signature", "unique"]):
                page_type = "specialties"
            else:
                page_type = "popular"

        file_path = get_page_path(wiki_base, section, page_type, slug)

        if slug in existing_pages:
            # Page exists — diff using LLM and prepare update
            existing_file_path = existing_pages[slug]
            try:
                existing_content = read_markdown(existing_file_path)
            except Exception as e:
                print(f"  Error reading existing file '{existing_file_path}': {e}")
                existing_content = ""

            if existing_content.strip():
                new_info = json.dumps({
                    "name": name,
                    "brand": product.get("brand", ""),
                    "category": product.get("category", ""),
                    "price": product.get("price", ""),
                    "specifications": product.get("specifications", {}),
                    "description": product.get("description", ""),
                }, indent=2)

                prompt = (
                    f"Compare NEW product data against the EXISTING wiki page content.\n\n"
                    f"EXISTING PAGE:\n{existing_content[:3000]}\n\n"
                    f"NEW DATA:\n{new_info}\n\n"
                    f"Identify:\n"
                    f"1. What information is genuinely new (not in existing page)\n"
                    f"2. What information contradicts existing content\n"
                    f"3. What information is unchanged\n\n"
                    f"Return JSON:\n"
                    f"{{\n"
                    f"  \"has_new_info\": true/false,\n"
                    f"  \"new_fields\": [\"field1\", \"field2\"],\n"
                    f"  \"contradictions\": [\n"
                    f"    {{\"field\": \"...\", \"existing_value\": \"...\", \"new_value\": \"...\"}}\n"
                    f"  ],\n"
                    f"  \"unchanged_fields\": [\"field1\", \"field2\"]\n"
                    f"}}"
                )

                try:
                    diff_result = call_llm_json(prompt, system="You are a knowledge diff analyzer.")

                    # Record contradictions
                    for conflict in diff_result.get("contradictions", []):
                        contradictions.append(Contradiction(
                            product_slug=slug,
                            field=conflict.get("field", "unknown"),
                            existing_value=conflict.get("existing_value", ""),
                            new_value=conflict.get("new_value", ""),
                            existing_source="existing wiki page",
                            new_source=source_id,
                            resolution="pending",
                            preferred_source="",
                        ))

                    if diff_result.get("has_new_info") or diff_result.get("contradictions"):
                        pages_to_update.append(WikiPage(
                            slug=slug,
                            title=name,
                            page_type=page_type,
                            section=section,
                            file_path=existing_file_path,
                            content="",  # Will be generated by update_pages node
                            sources=[source_id],
                            links=[],
                        ))
                except Exception as e:
                    print(f"  Error diffing '{slug}': {e}")
                    pages_to_update.append(WikiPage(
                        slug=slug,
                        title=name,
                        page_type=page_type,
                        section=section,
                        file_path=existing_file_path,
                        content="",
                        sources=[source_id],
                        links=[],
                    ))
            else:
                # Existing file was empty -> queue for update
                pages_to_update.append(WikiPage(
                    slug=slug,
                    title=name,
                    page_type=page_type,
                    section=section,
                    file_path=existing_file_path,
                    content="",
                    sources=[source_id],
                    links=[],
                ))
        else:
            # New product -> queue for creation
            pages_to_create.append(WikiPage(
                slug=slug,
                title=name,
                page_type=page_type,
                section=section,
                file_path=file_path,
                content="",  # Will be generated by create_pages node
                sources=[source_id],
                links=[],
            ))

    print(f"  Knowledge diff complete: {len(pages_to_create)} to create, {len(pages_to_update)} to update, {len(contradictions)} conflicts")

    return {
        "pages_to_create": pages_to_create,
        "pages_to_update": pages_to_update,
        "contradictions": contradictions,
    }


In [19]:
def _route_after_diff(state: WikiState) -> str:
    """
    Conditional routing after knowledge_diff.
    Routes to create_pages, update_pages, or both.
    If neither has items, skip to resolve_conflicts.
    """

    has_creates = bool(state.get("pages_to_create"))
    has_updates = bool(state.get("pages_to_update"))

    if has_creates and has_updates:
        return "create_pages"
    elif has_creates:
        return "create_pages"
    elif has_updates:
        return "update_pages"
    else:
        return "resolve_conflict"
        

def _route_after_create(state: WikiState) -> str:
    """After creating pages, check if we also need to update."""

    has_updates = bool(state.get("pages_to_update"))

    if has_updates:
        return "update_pages"
    else:
        return "resolve_conflict"


In [20]:
import json
from pathlib import Path
from datetime import datetime
from jinja2 import Template

from utils.file_io import write_markdown, get_page_path, slugify, page_exists
from utils.llm import call_llm, call_llm_json

def _load_template(template_name: str) -> Template:
    """Load a Jinja2 template from the templates directory."""
    template_dir = Path(__file__).parent.parent / "templates"
    template_path = template_dir / template_name
    if template_path.exists():
        return Template(template_path.read_text(encoding="utf-8"))
    # Fallback to a basic Markdown template if template file is missing
    return Template("# {{ name }}\n\n{{ description }}\n")


def _generate_product_page(product: dict, review_synthesis: dict | None, section: str) -> str:
    """Generate a product wiki page using Jinja2 template + LLM enrichment."""
    template = _load_template("product_page.md")

    # Build template context
    context = {
        "slug": product.get("slug", ""),
        "name": product.get("name", "Unknown Product"),
        "brand": product.get("brand", "Unknown"),
        "category": product.get("category", "Uncategorized"),
        "price": product.get("price", "N/A"),
        "currency": product.get("currency", "INR"),
        "original_price": None,
        "discount": None,
        "description": product.get("description", ""),
        "specifications": product.get("specifications", {}),
        "sentiment": None,
        "conflicts": [],
        "related_links": [],
        "sources": [product.get("source_file", "db_source")],
        "last_updated": datetime.now().strftime("%Y-%m-%d"),
    }

    # Add review synthesis if available
    if review_synthesis:
        context["sentiment"] = {
            "avg_rating": review_synthesis.get("avg_rating", 0),
            "total_reviews": review_synthesis.get("total_reviews", 0),
            "sentiment_summary": review_synthesis.get("sentiment_summary", ""),
            "top_pros": review_synthesis.get("top_pros", []),
            "top_cons": review_synthesis.get("top_cons", []),
            "best_reviews": review_synthesis.get("best_reviews", []),
        }

    # Enrich description using LLM if too short
    if len(context["description"]) < 50:
        specs_text = "\n".join(f"- {k}: {v}" for k, v in context["specifications"].items())
        prompt = (
            f"Write a concise 2-3 sentence product overview for '{context['name']}' "
            f"by {context['brand']}.\n\n"
            f"Category: {context['category']}\n"
            f"Price: {context['price']}\n"
            f"Specifications:\n{specs_text}\n\n"
            f"Write factually based only on the provided information. Do not invent features."
        )
        try:
            context["description"] = call_llm(prompt, system="You write concise product overviews.")
        except Exception as e:
            print(f"  Warning: LLM description enrichment failed for '{context['slug']}': {e}")

    return template.render(**context)


def _generate_marketing_page(product: dict, page_type: str) -> str:
    """Generate a marketing page (promotions/specialties/popular)."""
    if page_type == "promotions":
        template = _load_template("promotion_page.md")
    elif page_type == "popular":
        template = _load_template("popular_page.md")
    else:
        template = _load_template("promotion_page.md")

    # Filter out raw payload before sending to LLM
    product_info = json.dumps({k: v for k, v in product.items() if k != "raw_data"}, indent=2, default=str)

    prompt = (
        f"Generate marketing intelligence for this product:\n\n{product_info}\n\n"
        f"Page type: {page_type}\n\n"
        f"Return JSON with:\n"
        f"- why_promote: why this product should be promoted (2-3 sentences)\n"
        f"- selling_points: list of 3-5 key selling points\n"
        f"- target_audience: list of 2-3 target customer segments\n"
        f"- revenue_note: brief revenue impact note\n"
        f"- popularity_reason: why this is popular (if applicable)\n"
    )

    try:
        marketing = call_llm_json(prompt, system="You are a marketing strategist for e-commerce.")
    except Exception:
        marketing = {
            "why_promote": f"{product.get('name', 'Product')} is a strong offering in its category.",
            "selling_points": ["Quality product", "Competitive pricing"],
            "target_audience": ["General consumers"],
            "revenue_note": "Standard revenue potential.",
            "popularity_reason": "Solid product with positive engagement.",
        }

    prod_category = product.get("category", "")
    context = {
        "slug": product.get("slug", ""),
        "name": product.get("name", ""),
        "product_slug": product.get("slug", ""),
        "product_name": product.get("name", ""),
        "promotion_type": page_type.replace("_", " ").title(),
        "category": prod_category,
        "category_slug": slugify(prod_category) if prod_category else "general",
        "price": product.get("price", "N/A"),
        "currency": product.get("currency", "INR"),
        "rating": "N/A",
        "review_count": 0,
        "ranking": 0,
        "discount": None,
        "valid_until": None,
        "why_promote": marketing.get("why_promote", ""),
        "selling_points": marketing.get("selling_points", []),
        "target_audience": marketing.get("target_audience", []),
        "revenue_note": marketing.get("revenue_note", ""),
        "revenue_recommendation": marketing.get("revenue_note", ""),
        "popularity_reason": marketing.get("popularity_reason", ""),
        "customer_evidence": "",
        "customer_highlights": [],
        "pairings": [],
        "sources": [product.get("source_file", "db_source")],
        "last_updated": datetime.now().strftime("%Y-%m-%d"),
    }

    return template.render(**context)


def create_pages(state: WikiState) -> dict:
    """
    Create new wiki pages for all products in `pages_to_create`.
    """
    pages_to_create = state.get("pages_to_create", [])
    extracted_products = state.get("extracted_products", [])
    review_syntheses = state.get("review_syntheses", [])
    wiki_base = state.get("wiki_base_path", "")
    section = state.get("wiki_section", "knowledge")

    if not pages_to_create:
        return {"generated_pages": []}

    product_map = {p["slug"]: p for p in extracted_products if "slug" in p}
    review_map = {rs["product_slug"]: rs for rs in review_syntheses if "product_slug" in rs}

    generated: list[WikiPage] = []

    for page in pages_to_create:
        slug = page.get("slug")
        product = product_map.get(slug)

        if not product:
            continue

        # Generate page content
        if section == "knowledge":
            review_data = review_map.get(slug)
            content = _generate_product_page(product, review_data, section)
        else:
            content = _generate_marketing_page(product, page.get("page_type", "popular"))

        # Write markdown file
        file_path = page["file_path"]
        write_markdown(file_path, content)

        page_with_content = dict(page)
        page_with_content["content"] = content
        generated.append(page_with_content)

        print(f"  Created wiki page: {section}/{page.get('page_type', 'products')}/{slug}.md")

    # Generate Category pages for knowledge section
    if section == "knowledge":
        categories = set()
        for product in extracted_products:
            cat = product.get("category")
            if cat:
                categories.add(cat)

        for cat_name in categories:
            cat_slug = slugify(cat_name)

            if not page_exists(wiki_base, section, "categories", cat_slug):
                cat_products = [p for p in extracted_products if slugify(p.get("category", "")) == cat_slug]

                template = _load_template("category_page.md")
                context = {
                    "slug": cat_slug,
                    "name": cat_name,
                    "description": f"Products in the {cat_name} category.",
                    "products": [{
                        "slug": p.get("slug", ""),
                        "name": p.get("name", ""),
                        "brand": p.get("brand", ""),
                        "price": p.get("price", "N/A"),
                        "currency": p.get("currency", "INR"),
                        "rating": review_map.get(p.get("slug", ""), {}).get("avg_rating", "N/A"),
                        "review_count": review_map.get(p.get("slug", ""), {}).get("total_reviews", 0),
                    } for p in cat_products],
                    "top_rated": [],
                    "price_range": None,
                    "currency": "INR",
                    "trends": f"Category contains {len(cat_products)} products.",
                    "sources": list(set(p.get("source_file", "db_source") for p in cat_products)),
                    "last_updated": datetime.now().strftime("%Y-%m-%d"),
                }

                cat_content = template.render(**context)
                cat_path = get_page_path(wiki_base, section, "categories", cat_slug)
                write_markdown(cat_path, cat_content)
                print(f"  Created category page: {section}/categories/{cat_slug}.md")

    print(f"  Successfully created {len(generated)} new pages")
    return {"generated_pages": generated}


In [21]:
from datetime import datetime
from utils.file_io import read_markdown, write_markdown
from utils.llm import call_llm

def update_pages(state: WikiState) -> dict:
    """
    Update existing wiki pages with new information fetched from DB.
    Uses LLM to intelligently merge new data into existing content.
    """
    pages_to_update = state.get("pages_to_update", [])
    extracted_products = state.get("extracted_products", [])
    review_syntheses = state.get("review_syntheses", [])
    contradictions = state.get("contradictions", [])

    if not pages_to_update:
        return {"generated_pages": state.get("generated_pages", [])}

    # Build lookups
    product_map = {p.get("slug"): p for p in extracted_products if "slug" in p}
    review_map = {rs.get("product_slug"): rs for rs in review_syntheses if "product_slug" in rs}
    
    conflict_map: dict[str, list] = {}
    for c in contradictions:
        prod_slug = c.get("product_slug")
        if prod_slug:
            if prod_slug not in conflict_map:
                conflict_map[prod_slug] = []
            conflict_map[prod_slug].append(c)

    generated = list(state.get("generated_pages", []))

    for page in pages_to_update:
        slug = page.get("slug")
        file_path = page.get("file_path", "")
        
        try:
            existing_content = read_markdown(file_path)
        except Exception as e:
            print(f"  Error reading existing file '{file_path}': {e}")
            existing_content = ""

        product = product_map.get(slug)

        if not product or not existing_content:
            continue

        # Format the new information block
        new_info_parts = []
        new_info_parts.append(f"Product: {product.get('name', 'Unknown')}")
        new_info_parts.append(f"Brand: {product.get('brand', 'N/A')}")
        new_info_parts.append(f"Category: {product.get('category', 'N/A')}")
        new_info_parts.append(f"Price: {product.get('price', 'N/A')} {product.get('currency', 'INR')}")
        new_info_parts.append(f"Description: {product.get('description', 'N/A')}")

        specs = product.get("specifications", {})
        if specs:
            new_info_parts.append("Specifications:")
            for k, v in specs.items():
                new_info_parts.append(f"  - {k}: {v}")

        # Add review data if available
        review = review_map.get(slug)
        if review:
            new_info_parts.append(f"\nReview Summary:")
            new_info_parts.append(f"  Rating: {review.get('avg_rating', 'N/A')}/5 ({review.get('total_reviews', 0)} reviews)")
            new_info_parts.append(f"  Sentiment: {review.get('sentiment_summary', '')}")

        # Add conflicts if present
        conflicts = conflict_map.get(slug, [])
        if conflicts:
            new_info_parts.append("\nConflicts detected:")
            for c in conflicts:
                new_info_parts.append(f"  - {c.get('field')}: was '{c.get('existing_value')}', now '{c.get('new_value')}'")

        new_info = "\n".join(new_info_parts)
        source_id = product.get("source_file", "db_source")

        prompt = (
            f"You are updating an existing product wiki page with new information.\n\n"
            f"RULES:\n"
            f"1. Keep ALL existing content that is still valid\n"
            f"2. Integrate new information into appropriate sections\n"
            f"3. Do NOT remove existing source references\n"
            f"4. Add conflict blocks for contradictions (use > ⚠️ format)\n"
            f"5. Update the last_updated date to {datetime.now().strftime('%Y-%m-%d')}\n"
            f"6. Add the new source identifier ({source_id}) to the Sources section\n"
            f"7. Maintain the same markdown structure\n\n"
            f"EXISTING PAGE:\n{existing_content}\n\n"
            f"NEW INFORMATION:\n{new_info}\n\n"
            f"New source identifier: {source_id}\n\n"
            f"Return the COMPLETE updated page in markdown format."
        )

        try:
            updated_content = call_llm(
                prompt,
                system="You are a wiki page editor. Merge new data into existing pages carefully.",
                max_tokens=8192,
            )
            write_markdown(file_path, updated_content)

            page_with_content = dict(page)
            page_with_content["content"] = updated_content
            generated.append(page_with_content)

            print(f"  Updated wiki page: {page.get('section')}/{page.get('page_type')}/{slug}.md")
        except Exception as e:
            print(f"  Error updating wiki page for '{slug}': {e}")

    print(f"  Successfully updated {len(pages_to_update)} existing pages")
    return {"generated_pages": generated}


In [22]:

from utils.llm import call_llm_json

def resolve_conflict(state: WikiState) -> dict:
    """
    Process detected contradictions from state.
    For each conflict, determine:
      1. Which source should be preferred
      2. Record resolution or mark as pending
    """
    contradictions = state.get("contradictions", [])

    if not contradictions:
        print("  No conflicts to resolve")
        return {"contradictions": []}

    resolved: list[Contradiction] = []

    for conflict in contradictions:
        prod_slug = conflict.get("product_slug", "unknown")
        field_name = conflict.get("field", "unknown")
        exist_val = conflict.get("existing_value", "")
        exist_src = conflict.get("existing_source", "existing wiki")
        new_val = conflict.get("new_value", "")
        new_src = conflict.get("new_source", "db_source")

        prompt = (
            f"A data conflict was detected for product '{prod_slug}'.\n\n"
            f"Field: {field_name}\n"
            f"Existing value: {exist_val} (Source: {exist_src})\n"
            f"New value: {new_val} (Source: {new_src})\n\n"
            f"Which source should be preferred? Consider:\n"
            f"- Official DB/manufacturer specs are usually more accurate than older entries\n"
            f"- More recent data may reflect updates\n"
            f"- If unable to determine with confidence, mark as 'pending'\n\n"
            f"Return JSON: {{\n"
            f"  \"resolution\": \"resolved\" or \"pending\",\n"
            f"  \"preferred_source\": \"which source to prefer\",\n"
            f"  \"reasoning\": \"why\"\n"
            f"}}"
        )

        try:
            result = call_llm_json(prompt, system="You are a data quality analyst.")
            conflict_resolved = dict(conflict)
            conflict_resolved["resolution"] = result.get("resolution", "pending")
            conflict_resolved["preferred_source"] = result.get("preferred_source", new_src)
            resolved.append(conflict_resolved)
        except Exception as e:
            print(f"  Error resolving conflict for '{prod_slug}': {e}")
            resolved.append(conflict)

    pending_count = sum(1 for c in resolved if c.get("resolution") == "pending")
    resolved_count = len(resolved) - pending_count
    print(f"  Conflicts processed: {resolved_count} resolved, {pending_count} pending")

    return {"contradictions": resolved}


In [23]:
import re
import os
from pathlib import Path

from utils.file_io import read_markdown, write_markdown, list_wiki_pages
from utils.llm import call_llm_json

def cross_reference(state: WikiState) -> dict:
    """
    Add cross-references (wiki-links) between related pages.
    """
    wiki_base = state.get("wiki_base_path", "")
    section = state.get("wiki_section", "knowledge")
    generated_pages = state.get("generated_pages", [])
    extracted_products = state.get("extracted_products", [])

    if not generated_pages:
        return {}

    wiki_dir = f"{wiki_base}/wiki"

    # Get all existing pages across sections
    try:
        knowledge_pages = list_wiki_pages(wiki_dir, "knowledge")
        marketing_pages = list_wiki_pages(wiki_dir, "marketing")
    except Exception as e:
        print(f"  Error reading wiki pages for cross-referencing: {e}")
        knowledge_pages, marketing_pages = {}, {}

    # Build context of available pages
    all_pages = {}
    wiki_path_obj = Path(wiki_dir)

    for slug, path in knowledge_pages.items():
        try:
            rel_path = str(Path(path).relative_to(wiki_path_obj))
        except Exception:
            rel_path = path
        all_pages[slug] = {"path": path, "rel": rel_path, "section": "knowledge"}

    for slug, path in marketing_pages.items():
        try:
            rel_path = str(Path(path).relative_to(wiki_path_obj))
        except Exception:
            rel_path = path
        all_pages[slug] = {"path": path, "rel": rel_path, "section": "marketing"}

    if not all_pages:
        return {}

    product_map = {p.get("slug"): p for p in extracted_products if "slug" in p}
    available_pages_text = "\n".join(f"- {slug} ({info['section']}/{info['rel']})" for slug, info in all_pages.items())

    for page in generated_pages:
        slug = page.get("slug")
        file_path = page.get("file_path")

        if not slug or not file_path:
            continue

        try:
            content = read_markdown(file_path)
        except Exception as e:
            print(f"  Error reading file for cross-referencing '{file_path}': {e}")
            content = ""

        if not content.strip():
            continue

        product = product_map.get(slug, {})

        prompt = (
            f"Given this wiki page and the list of all available pages, "
            f"identify which pages should be cross-linked.\n\n"
            f"CURRENT PAGE: {slug}\n"
            f"Category: {product.get('category', 'unknown')}\n"
            f"Brand: {product.get('brand', 'unknown')}\n\n"
            f"AVAILABLE PAGES:\n{available_pages_text[:3000]}\n\n"
            f"Return JSON: {{\n"
            f"  \"related_pages\": [\n"
            f"    {{\"slug\": \"...\", \"relationship\": \"category/competitor/related/promotion\"}}\n"
            f"  ]\n"
            f"}}"
        )

        try:
            result = call_llm_json(prompt, system="You are a knowledge graph curator.")
            related = result.get("related_pages", [])

            if related:
                # Add a ## Related section if not already present
                if "## Related" not in content:
                    links_md = "\n## Related\n\n"
                    for rel in related:
                        rel_slug = rel.get("slug", "")
                        relationship = rel.get("relationship", "related")
                        if rel_slug in all_pages and rel_slug != slug:
                            rel_info = all_pages[rel_slug]
                            try:
                                rel_link_path = os.path.relpath(rel_info['path'], start=os.path.dirname(file_path)).replace("\\", "/")
                            except Exception:
                                rel_link_path = rel_info['path']
                            display_title = rel_slug.replace("-", " ").title()
                            links_md += f"- [{display_title}]({rel_link_path}) ({relationship})\n"

                    content += links_md
                    write_markdown(file_path, content)
                    print(f"  Cross-linked: '{slug}' -> {len(related)} related pages")

        except Exception as e:
            print(f"  Error cross-referencing '{slug}': {e}")

    print("  Cross-referencing complete.")
    return {}


In [24]:
from pathlib import Path
from datetime import datetime

from utils.file_io import write_markdown, list_wiki_pages, read_markdown

def _extract_title_from_md(content: str) -> str:
    """Extract the first H1 heading (# Heading) from markdown content."""
    if not content:
        return "Untitled"
    for line in content.split("\n"):
        if line.startswith("# "):
            return line[2:].strip()
    return "Untitled"


def _extract_metadata_field(content: str, field: str) -> str:
    """Extract a field from YAML frontmatter."""
    if not content:
        return ""
    in_frontmatter = False
    for line in content.split("\n"):
        if line.strip() == "---":
            if in_frontmatter:
                break
            in_frontmatter = True
            continue
        if in_frontmatter and line.startswith(f"{field}:"):
            return line.split(":", 1)[1].strip()
    return ""


def update_index(state: WikiState) -> dict:
    """
    Rebuild index.md files:
    1. Section-level index (knowledge/index.md or marketing/index.md)
    2. Master index (wiki/index.md)
    """
    wiki_base = state.get("wiki_base_path", "")
    section = state.get("wiki_section", "knowledge")
    wiki_dir = f"{wiki_base}/wiki"

    # ── Section Index ────────────────────────────────────────────────────
    try:
        section_pages = list_wiki_pages(wiki_dir, section)
    except Exception as e:
        print(f"  Error listing section pages for '{section}': {e}")
        section_pages = {}

    # Group pages by page_type (folder name)
    pages_by_type: dict[str, list[dict]] = {}
    for slug, path in section_pages.items():
        page_type = Path(path).parent.name
        try:
            content = read_markdown(path)
        except Exception:
            content = ""

        title = _extract_title_from_md(content)

        if page_type not in pages_by_type:
            pages_by_type[page_type] = []

        pages_by_type[page_type].append({
            "slug": slug,
            "title": title if title != "Untitled" else slug.replace("-", " ").title(),
            "path": path,
            "page_type": page_type,
        })

    index_title = "Product Knowledge Index" if section == "knowledge" else "Marketing & Promotions Index"

    index_lines = [
        f"# {index_title}",
        "",
        f"_Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M')}_",
        "",
    ]

    type_display = {
        "products": "Products",
        "categories": "Categories",
        "reviews": "Reviews",
        "insights": "Insights",
        "promotions": "Promotions",
        "specialties": "Specialties",
        "popular": "Popular Items",
        "campaigns": "Campaigns",
    }

    for page_type in sorted(pages_by_type.keys()):
        pages = pages_by_type[page_type]
        display = type_display.get(page_type, page_type.title())
        index_lines.append(f"## {display}")
        index_lines.append("")

        for page in sorted(pages, key=lambda x: x["title"]):
            link = f"[{page['title']}](./{page_type}/{page['slug']}.md)"
            index_lines.append(f"- {link}")

        index_lines.append("")

    section_index_path = f"{wiki_dir}/{section}/index.md"
    write_markdown(section_index_path, "\n".join(index_lines))

    # Master Index 
    master_lines = [
        "# Merchant Knowledge Index",
        "",
        f"_Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M')}_",
        "",
    ]

    for s in ["knowledge", "marketing"]:
        try:
            s_pages = list_wiki_pages(wiki_dir, s)
        except Exception:
            s_pages = {}

        if not s_pages:
            continue

        s_title = "Knowledge Base" if s == "knowledge" else "Marketing Intelligence"
        master_lines.append(f"## {s_title}")
        master_lines.append("")

        s_by_type: dict[str, list] = {}
        for slug, path in s_pages.items():
            pt = Path(path).parent.name
            if pt not in s_by_type:
                s_by_type[pt] = []
            try:
                content = read_markdown(path)
            except Exception:
                content = ""
            title = _extract_title_from_md(content)
            display_title = title if title != "Untitled" else slug.replace("-", " ").title()
            s_by_type[pt].append({"slug": slug, "title": display_title})

        for pt in sorted(s_by_type.keys()):
            display = type_display.get(pt, pt.title())
            master_lines.append(f"### {display}")
            master_lines.append("")
            for page in sorted(s_by_type[pt], key=lambda x: x["title"]):
                master_lines.append(f"- [{page['title']}](./{s}/{pt}/{page['slug']}.md)")
            master_lines.append("")

    master_index_path = f"{wiki_dir}/index.md"
    write_markdown(master_index_path, "\n".join(master_lines))

    total_pages = len(section_pages)
    print(f"  Index updated: {total_pages} pages indexed in '{section}'")

    return {
        "index_updates": [{"section": section, "page_count": total_pages}],
    }


In [25]:
from datetime import datetime
from utils.file_io import append_markdown
from utils.markdown_builder import log_entry_block


def append_log(state: WikiState) -> dict:
    """
    Append an operation record to wiki/log.md.
    """
    wiki_base = state.get("wiki_base_path", "")
    section = state.get("wiki_section", "knowledge")
    collected_data = state.get("collected_data", [])
    pages_to_create = state.get("pages_to_create", [])
    pages_to_update = state.get("pages_to_update", [])
    contradictions = state.get("contradictions", [])
    extracted_reviews = state.get("extracted_reviews", [])
    validation_errors = state.get("validation_errors", [])

    # Format DB record identifiers for log trace
    sources_processed = [
        f"db_record_{idx + 1} ({item.get('category', 'data')})" 
        for idx, item in enumerate(collected_data)
    ]

    pages_created = [f"{p.get('page_type', 'products')}/{p.get('slug', '')}.md" for p in pages_to_create if "slug" in p]
    pages_updated = [f"{p.get('page_type', 'products')}/{p.get('slug', '')}.md" for p in pages_to_update if "slug" in p]

    status = "SUCCESS"
    if validation_errors:
        status = "PARTIAL"
    if state.get("status") == "failed":
        status = "FAILED"

    log_block = log_entry_block(
        operation="ingest",
        section=section,
        files=sources_processed,
        products_added=len(pages_to_create),
        products_updated=len(pages_to_update),
        reviews_processed=len(extracted_reviews),
        pages_created=pages_created,
        pages_updated=pages_updated,
        conflicts=len(contradictions),
        status=status,
        errors=validation_errors,
    )

    log_path = f"{wiki_base}/wiki/log.md"
    try:
        append_markdown(log_path, log_block)
    except Exception as e:
        print(f"  Error appending to log file '{log_path}': {e}")

    # Build the LogEntry object for state
    log_entry = LogEntry(
        timestamp=datetime.now().isoformat(),
        operation="ingest",
        section=section,
        files_processed=sources_processed,
        products_added=len(pages_to_create),
        products_updated=len(pages_to_update),
        reviews_processed=len(extracted_reviews),
        pages_created=pages_created,
        pages_updated=pages_updated,
        conflicts=len(contradictions),
        status=status,
        errors=validation_errors or [],
    )

    print(f"  Log appended: status={status}")
    return {"log_entry": log_entry}


In [26]:
"""
Performs wiki health checks (lint):
  - Orphan pages (not listed in index.md)
  - Duplicate products
  - Missing reviews
  - Conflicting specs
  - Broken cross-links
"""

import os
import re
from pathlib import Path
from collections import Counter

from utils.file_io import list_wiki_pages, read_markdown

def validate_wiki(state: WikiState) -> dict:
    """
    Validate the wiki for health issues and broken links.
    Returns a ValidationResult object with overall health score.
    """
    wiki_base = state.get("wiki_base_path", "")
    wiki_dir = f"{wiki_base}/wiki"

    orphan_pages: list[str] = []
    duplicate_products: list[str] = []
    missing_reviews: list[str] = []
    conflicting_specs: list[str] = []
    broken_links: list[str] = []
    outdated_prices: list[str] = []

    # Check both sections: knowledge and marketing
    for section in ["knowledge", "marketing"]:
        try:
            pages = list_wiki_pages(wiki_dir, section)
        except Exception as e:
            print(f"  Error reading wiki pages for section '{section}': {e}")
            pages = {}

        # Read the section index file
        index_path = f"{wiki_dir}/{section}/index.md"
        try:
            index_content = read_markdown(index_path)
        except Exception:
            index_content = ""

        # ── Check for orphan pages ──────────────────────────────────────
        for slug, path in pages.items():
            if slug not in index_content:
                orphan_pages.append(f"{section}/{slug}")

        # ── Check for duplicate product names ───────────────────────────
        if section == "knowledge":
            product_pages = {s: p for s, p in pages.items() if "products" in p}
            titles = []
            for slug, path in product_pages.items():
                try:
                    content = read_markdown(path)
                except Exception:
                    content = ""

                for line in content.split("\n"):
                    if line.startswith("# "):
                        titles.append(line[2:].strip().lower())
                        break

            title_counts = Counter(titles)
            for title, count in title_counts.items():
                if count > 1:
                    duplicate_products.append(f"{title} (appears {count} times)")

        # ── Check for products missing reviews ──────────────────────────
        if section == "knowledge":
            product_slugs = {s for s, p in pages.items() if "products" in p}

            for slug in product_slugs:
                if slug not in pages:
                    continue
                try:
                    content = read_markdown(pages[slug])
                except Exception:
                    content = ""

                if "Customer Sentiment" in content:
                    if "No review data available" in content:
                        missing_reviews.append(slug)
                elif "review" not in content.lower() and "rating" not in content.lower():
                    missing_reviews.append(slug)

        # ── Check for broken links ──────────────────────────────────────
        for slug, path in pages.items():
            try:
                content = read_markdown(path)
            except Exception:
                continue

            # Standard markdown links: [label](target.md)
            standard_links = re.findall(r'\[([^\]]+)\]\(([^)]+)\)', content)
            for label, link in standard_links:
                if link.startswith("http") or link.startswith("#"):
                    continue

                page_dir = os.path.dirname(path)
                target_path = os.path.abspath(os.path.join(page_dir, link))
                if not os.path.exists(target_path):
                    broken_links.append(f"{slug} -> {link}")

            # Double bracket links: [[...]]
            bracket_links = re.findall(r'\[\[([^\]]+)\]\]', content)
            for link in bracket_links:
                link_slug = link.split("|")[0].strip()
                link_parts = link_slug.replace("\\", "/").split("/")
                target_slug = link_parts[-1] if link_parts else link_slug

                try:
                    all_pages = {}
                    all_pages.update(list_wiki_pages(wiki_dir, "knowledge"))
                    all_pages.update(list_wiki_pages(wiki_dir, "marketing"))
                except Exception:
                    all_pages = pages

                if target_slug not in all_pages and target_slug not in pages:
                    broken_links.append(f"{slug} -> {link_slug}")

        # ── Check for conflict markers ──────────────────────────────────
        for slug, path in pages.items():
            try:
                content = read_markdown(path)
            except Exception:
                continue

            if "Data Conflict" in content:
                conflicting_specs.append(slug)

    # ── Calculate health score ──────────────────────────────────────────
    total_issues = (
        len(orphan_pages) +
        len(duplicate_products) +
        len(missing_reviews) +
        len(broken_links) +
        len(conflicting_specs)
    )

    try:
        all_pages = list_wiki_pages(wiki_dir, "knowledge")
        all_pages.update(list_wiki_pages(wiki_dir, "marketing"))
    except Exception:
        all_pages = {}

    total_pages = max(len(all_pages), 1)
    health_score = max(0.0, 1.0 - (total_issues / (total_pages * 3)))

    # Print Wiki Health Report
    print(f"\n  ---- Wiki Health Report ----")
    print(f"  Orphan pages: {len(orphan_pages)}")
    print(f"  Duplicate products: {len(duplicate_products)}")
    print(f"  Conflicting specs: {len(conflicting_specs)}")
    print(f"  Missing reviews: {len(missing_reviews)}")
    print(f"  Broken links: {len(broken_links)}")
    print(f"  Health score: {health_score:.0%}")
    print(f"  ----------------------------\n")

    result = ValidationResult(
        orphan_pages=orphan_pages,
        duplicate_products=duplicate_products,
        missing_reviews=missing_reviews,
        conflicting_specs=conflicting_specs,
        outdated_prices=outdated_prices,
        broken_links=broken_links,
        health_score=health_score,
    )

    return {
        "validation_result": result,
        "validation_errors": orphan_pages + duplicate_products + broken_links,
        "status": "success",
    }


In [1]:
graph_builder = StateGraph(WikiState)

# nodes
graph_builder.add_node("collect_data", collect_data)
graph_builder.add_node("extract_entities", extract_entities)
graph_builder.add_node("extract_reviews", extract_reviews)
graph_builder.add_node("search_existing_wiki", search_existing_wiki)
graph_builder.add_node("knowledge_diff", knowledge_diff)
graph_builder.add_node("create_pages", create_pages)
graph_builder.add_node("update_pages", update_pages)
graph_builder.add_node("resolve_conflict", resolve_conflict)
graph_builder.add_node("cross_reference", cross_reference)
graph_builder.add_node("update_index", update_index)
graph_builder.add_node("append_log", append_log)
graph_builder.add_node("validate_wiki", validate_wiki)


# edges
graph_builder.add_edge(START, "collect_data")
graph_builder.add_edge("collect_data", "extract_entities")
graph_builder.add_edge("extract_entities", "extract_reviews")
graph_builder.add_edge("extract_reviews", "search_existing_wiki")
graph_builder.add_edge("search_existing_wiki", "knowledge_diff")
graph_builder.add_conditional_edges(
    "knowledge_diff",
     _route_after_diff , 
    {
       "create_pages": "create_pages",
       "update_pages": "update_pages",
       "resolve_conflict": "resolve_conflict"
    }
)

graph_builder.add_conditional_edges(
    "create_pages",
    _route_after_create,
    {
        "update_pages" : "update_pages",
        "resolve_conflict": "resolve_conflict"
    }
)
graph_builder.add_edge("update_pages", "resolve_conflict")
graph_builder.add_edge("resolve_conflict", "cross_reference")
graph_builder.add_edge("cross_reference", "update_index")
graph_builder.add_edge( "update_index", "append_log")
graph_builder.add_edge("append_log", "validate_wiki")


workflow = graph_builder.compile()
workflow

NameError: name 'StateGraph' is not defined

In [28]:
from pathlib import Path
def list_wiki_pages(wiki_dir: str, section: str) -> dict[str, str]:
    """
    List all .md pages in a wiki section.
    Returns dict of slug -> file_path.
    """
    pages = {}
    section_dir = Path(wiki_dir) / section
    if not section_dir.exists():
        return pages

    for md_file in section_dir.rglob("*.md"):
        if md_file.name in ("index.md", "log.md"):
            continue
        slug = md_file.stem
        pages[slug] = str(md_file)
    return pages

In [29]:
res = list_wiki_pages(r"C:\Users\ps302\OneDrive\Desktop\Razorpay\backend\merchant_knowledge\wiki", "knowledge")

In [30]:
for item in res:
    print(res)

{'headphones': 'C:\\Users\\ps302\\OneDrive\\Desktop\\Razorpay\\backend\\merchant_knowledge\\wiki\\knowledge\\categories\\headphones.md', 'smartphones': 'C:\\Users\\ps302\\OneDrive\\Desktop\\Razorpay\\backend\\merchant_knowledge\\wiki\\knowledge\\categories\\smartphones.md', 'tablets': 'C:\\Users\\ps302\\OneDrive\\Desktop\\Razorpay\\backend\\merchant_knowledge\\wiki\\knowledge\\categories\\tablets.md', 'apple-airpods-pro-2': 'C:\\Users\\ps302\\OneDrive\\Desktop\\Razorpay\\backend\\merchant_knowledge\\wiki\\knowledge\\products\\apple-airpods-pro-2.md', 'ipad-air-m2': 'C:\\Users\\ps302\\OneDrive\\Desktop\\Razorpay\\backend\\merchant_knowledge\\wiki\\knowledge\\products\\ipad-air-m2.md', 'iphone-15-pro': 'C:\\Users\\ps302\\OneDrive\\Desktop\\Razorpay\\backend\\merchant_knowledge\\wiki\\knowledge\\products\\iphone-15-pro.md', 'iphone-15': 'C:\\Users\\ps302\\OneDrive\\Desktop\\Razorpay\\backend\\merchant_knowledge\\wiki\\knowledge\\products\\iphone-15.md', 'oneplus-12': 'C:\\Users\\ps302\\On

In [31]:
#  tool binding to the llm
# llm_with_tools = llm.bind_tools(tools)

In [32]:
# Prompt
SYSTEM_PROMPT = """
You are the Merchant Commerce Agent.

You are responsible for handling customer shopping requests.

You have access to commerce tools.

Rules:

1. Never claim that a product was added unless add_to_cart
   successfully confirms it.

2. Never directly modify the cart yourself.

3. Use get_cart when you need current cart information.

4. Use get_upsell_products when the customer asks for
   recommendations or when an appropriate complementary
   product can be suggested.

5. Do not invent products.

6. Keep responses concise and commerce-focused.

7. If a tool fails, clearly tell the customer that the
   requested operation could not be completed.

8. MUST: Do not add the item to the cart until and unless customer ask to do, if he is asking about the product just tell it  where you have or not,
    and then ask him if he would like it to be added to the cart and along with it recommend the items that this inventory have
"""

In [33]:
# class CommerceState(TypedDict):

#     messages: Annotated[ list, add_messages ]

# """ later add ons
# user_id
# merchant_id
# cart
# payment_context
# policy_context
# """

In [34]:
# def merchant_llm_node(state: CommerceState):

#     messages = state["messages"]

#     system_message = {
#         "role": "system",
#         "content": SYSTEM_PROMPT
#     }

#     response = llm_with_tools.invoke(
#         [system_message] + messages
#     )

#     return {
#         "messages": [response]
#     }

    

In [35]:
# tool_node = ToolNode(tools)

In [36]:
# graph = StateGraph(CommerceState)

# # nodes
# graph.add_node("merchant_llm", merchant_llm_node )
# graph.add_node( "tools", tool_node )

# # edges
# graph.add_edge(START, "merchant_llm" )
# graph.add_conditional_edges("merchant_llm", tools_condition)
# graph.add_edge("tools", "merchant_llm")


# workflow = graph.compile()

In [37]:
class WikiState(TypedDict):
    """
    Central state that flows through all LangGraph nodes.

    Both 'knowledge' and 'marketing' sections use this same state.
    The `wiki_section` field controls which sub-directory is targeted.
    """

  

     # slug -> file_path
  